# Table of Contents

1. Pulling the Data
1. Preparing the Data
1. Further Cleaning

# Pulling the Data

All of the data collected from others was collected

In [1]:
# !pip install gspread pandas google-auth
import gspread
from google.oauth2.service_account import Credentials
import os, pandas as pd
from pathlib import Path


In [2]:
os.environ["GOOGLE_SHEETS_CREDENTIALS"] = (
    "/opt/notebooks/psalms_nlp_sp26/private/psalms-blind-scoring-da42a38adf3c.json"
)

os.getcwd()

'/opt/notebooks/psalms_nlp_sp26/query_compare'

In [3]:
# Current notebook directory
notebook_dir = Path.cwd()

# Build path to the JSON
cred_path = notebook_dir.parent / "private" / "psalms-blind-scoring-da42a38adf3c.json"
print("Credential path exists?", cred_path.exists())

os.getcwd()

Credential path exists? True


'/opt/notebooks/psalms_nlp_sp26/query_compare'

In [4]:
import socket
socket.gethostbyname("oauth2.googleapis.com")


'172.253.115.95'

In [5]:
creds = Credentials.from_service_account_file(
    os.environ["GOOGLE_SHEETS_CREDENTIALS"],
    scopes=[
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive"
    ]
)

client = gspread.authorize(creds)

sheet = client.open("results_scored")

In [6]:
# Access second sheet (index 1)
worksheet2 = sheet.get_worksheet(1)

# Get all values
data = worksheet2.get_all_values()

# Convert to DataFrame (first row as header)
df = pd.DataFrame(data[1:], columns=data[0])

# storing the collected data for reference later
df.to_csv("../data/results_scored.csv", index=False, mode="w")


# Preparing the data
I want each row to hold one of the four possible scored for each of the `236 results`. So if rebuilt properly, we should end up with a total of: 
$$236 * 4 = 944\ results$$

I need to start by unpivoting my own score separte from the other scores.

## Numbering the Results 
I also want to be able to reference the order of the results within each search. I collected the top 5 results from each search. There was a bug in my code that took the top 6 results from searches. I am going to just worry about the top 5 results to keep everything fair. 

In [7]:
# temporary dataframe to not break the original 
temp = df.copy()

# aqdding a column to number the indivudal results
temp['numbered_result'] = pd.NA
temp

,Query,Method,Similarity Score (%),Text,Psalm Num,Verse,CadenScore,Score1,User1,Score2,User2,Score3,User3,numbered_result
0,Create in me a clean heart,TFIDF_GLoVe,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,9,5,p06,1,p03,3,p08,NaN
1,Create in me a clean heart,TFIDF_GLoVe,25.67,Bible,4,For the End in psalms an ode by David You hear...,6,1,p03,8,p04,,,NaN
2,Create in me a clean heart,TFIDF_GLoVe,21.90,Bible,31,By David concerning understanding Blessed are ...,3,7,p06,1,p03,,,NaN
3,Create in me a clean heart,TFIDF_GLoVe,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...,10,7,p06,,,,,NaN
4,Create in me a clean heart,TFIDF_GLoVe,18.12,Bible,61,For the End for Jeduthun a psalm by David Shal...,7,2,p01,0,p03,2,p08,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,Verses where the psalmist remembers past deliv...,TFIDF,9.98,Psalter,23,"The earth is the Lord's, and the fulness there...",2,0,p01,0,p06,7,p04,NaN
232,Verses where the psalmist remembers past deliv...,TFIDF,7.79,Bible,130,1An ode of ascents by David OLord My heart is ...,1,0,p03,0,p17,,,NaN
233,Verses where the psalmist remembers past deliv...,TFIDF,7.08,Psalter,61,Shall not my soul be subject unto God? for fro...,3,10,p06,,,,,NaN
234,Verses where the psalmist remembers past deliv...,TFIDF,5.61,Psalter,35,"The transgressor, that he may sin, saith withi...",8,2,p10,7,p05,,,NaN


In [8]:
n = temp.shape[0]

# starting with the number 1 result of a query
num_result = 1

query = temp["Query"].iloc[0]
method = temp["Method"].iloc[0]

for i in range(n):
    # checking if we are in the same group fo data to be numbered
    if query == temp["Query"].iloc[i] and method == temp["Method"].iloc[i]:
        temp["numbered_result"].iloc[i] = num_result
        num_result += 1
        
    # in an new group of results
    else:
        # the current result is the number result for the new set of results
        temp["numbered_result"].iloc[i] = 1
        # reset the number result
        num_result = 2
        # update to the new target query & method
        query = temp["Query"].iloc[i]
        method = temp["Method"].iloc[i]
        
temp.tail(20)


,Query,Method,Similarity Score (%),Text,Psalm Num,Verse,CadenScore,Score1,User1,Score2,User2,Score3,User3,numbered_result
216,Verses where the psalmist remembers past deliv...,TFIDF_GLoVe,21.41,Psalter,59,"O God, Thou hast cast us off, and hast destroy...",2,8,p01,6,p03,5,p06,3
217,Verses where the psalmist remembers past deliv...,TFIDF_GLoVe,20.54,Bible,61,For the End for Jeduthun a psalm by David Shal...,8,1,p03,9,p01,,,4
218,Verses where the psalmist remembers past deliv...,TFIDF_GLoVe,20.02,Psalter,103,"Bless the Lord, O my soul. O Lord my God, Thou...",4,4,p01,0,p06,0,p03,5
219,Verses where the psalmist remembers past deliv...,TFIDF_GLoVe,19.50,Psalter,23,"The earth is the Lord's, and the fulness there...",4,1,p01,1,p09,0,p06,6
220,Verses where the psalmist remembers past deliv...,BERT,99.88,Bible,32,By David Rejoice greatly in the Lord O righteo...,1,6,p01,7,p05,2,p06,1
221,Verses where the psalmist remembers past deliv...,BERT,99.87,Bible,8,For the End concerning the winepresses a psalm...,9,2,p01,0,p03,3,p04,2
222,Verses where the psalmist remembers past deliv...,BERT,99.87,Bible,17,1For the End by the child of the Lord David wh...,10,9,p01,,,,,3
223,Verses where the psalmist remembers past deliv...,BERT,99.87,Bible,22,A psalm by David The Lord is my shepherd I sha...,7,3,p01,8,p09,,,4
224,Verses where the psalmist remembers past deliv...,BERT,99.87,Bible,34,By David OLord judge those who injure me Make ...,8,2,p02,9,p01,,,5
225,Verses where the psalmist remembers past deliv...,SBERT,97.02,Psalter,87,"O Lord God of my salvation, I have cried by da...",3,2,p01,2,p03,0,p04,1


In [9]:
# select first 7 columns + last column
cols_to_keep = list(temp.columns[:7]) + [temp.columns[-1]]
caden = temp[cols_to_keep]

caden.head()


,Query,Method,Similarity Score (%),Text,Psalm Num,Verse,CadenScore,numbered_result
0,Create in me a clean heart,TFIDF_GLoVe,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,9,1
1,Create in me a clean heart,TFIDF_GLoVe,25.67,Bible,4,For the End in psalms an ode by David You hear...,6,2
2,Create in me a clean heart,TFIDF_GLoVe,21.90,Bible,31,By David concerning understanding Blessed are ...,3,3
3,Create in me a clean heart,TFIDF_GLoVe,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...,10,4
4,Create in me a clean heart,TFIDF_GLoVe,18.12,Bible,61,For the End for Jeduthun a psalm by David Shal...,7,5


In [10]:
caden['User'] = 'caden'

caden = caden.rename(columns={"CadenScore": "Score"})

caden

/tmp/ipykernel_167/3532660364.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  caden['User'] = 'caden'


,Query,Method,Similarity Score (%),Text,Psalm Num,Verse,Score,numbered_result,User
0,Create in me a clean heart,TFIDF_GLoVe,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,9,1,caden
1,Create in me a clean heart,TFIDF_GLoVe,25.67,Bible,4,For the End in psalms an ode by David You hear...,6,2,caden
2,Create in me a clean heart,TFIDF_GLoVe,21.90,Bible,31,By David concerning understanding Blessed are ...,3,3,caden
3,Create in me a clean heart,TFIDF_GLoVe,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...,10,4,caden
4,Create in me a clean heart,TFIDF_GLoVe,18.12,Bible,61,For the End for Jeduthun a psalm by David Shal...,7,5,caden
...,...,...,...,...,...,...,...,...,...
231,Verses where the psalmist remembers past deliv...,TFIDF,9.98,Psalter,23,"The earth is the Lord's, and the fulness there...",2,2,caden
232,Verses where the psalmist remembers past deliv...,TFIDF,7.79,Bible,130,1An ode of ascents by David OLord My heart is ...,1,3,caden
233,Verses where the psalmist remembers past deliv...,TFIDF,7.08,Psalter,61,Shall not my soul be subject unto God? for fro...,3,4,caden
234,Verses where the psalmist remembers past deliv...,TFIDF,5.61,Psalter,35,"The transgressor, that he may sin, saith withi...",8,5,caden


In [11]:
caden  = caden[['Query', 'Method', "numbered_result", 'Similarity Score (%)', 'Text', 'Psalm Num', 'Verse', 'User', 'Score']]
caden

,Query,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,caden,9
1,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,caden,6
2,Create in me a clean heart,TFIDF_GLoVe,3,21.90,Bible,31,By David concerning understanding Blessed are ...,caden,3
3,Create in me a clean heart,TFIDF_GLoVe,4,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...,caden,10
4,Create in me a clean heart,TFIDF_GLoVe,5,18.12,Bible,61,For the End for Jeduthun a psalm by David Shal...,caden,7
...,...,...,...,...,...,...,...,...,...
231,Verses where the psalmist remembers past deliv...,TFIDF,2,9.98,Psalter,23,"The earth is the Lord's, and the fulness there...",caden,2
232,Verses where the psalmist remembers past deliv...,TFIDF,3,7.79,Bible,130,1An ode of ascents by David OLord My heart is ...,caden,1
233,Verses where the psalmist remembers past deliv...,TFIDF,4,7.08,Psalter,61,Shall not my soul be subject unto God? for fro...,caden,3
234,Verses where the psalmist remembers past deliv...,TFIDF,5,5.61,Psalter,35,"The transgressor, that he may sin, saith withi...",caden,8


Moving on to prepaering the external scores.

In [12]:
external = temp[['Query', 'Method', "numbered_result", 'Similarity Score (%)', 'Text', 'Psalm Num', 'Verse', 'User1', 'Score1', 'User2', 'Score2', 'User3', 'Score3']]

external

,Query,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User1,Score1,User2,Score2,User3,Score3
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,p06,5,p03,1,p08,3
1,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,p03,1,p04,8,,
2,Create in me a clean heart,TFIDF_GLoVe,3,21.90,Bible,31,By David concerning understanding Blessed are ...,p06,7,p03,1,,
3,Create in me a clean heart,TFIDF_GLoVe,4,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...,p06,7,,,,
4,Create in me a clean heart,TFIDF_GLoVe,5,18.12,Bible,61,For the End for Jeduthun a psalm by David Shal...,p01,2,p03,0,p08,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,Verses where the psalmist remembers past deliv...,TFIDF,2,9.98,Psalter,23,"The earth is the Lord's, and the fulness there...",p01,0,p06,0,p04,7
232,Verses where the psalmist remembers past deliv...,TFIDF,3,7.79,Bible,130,1An ode of ascents by David OLord My heart is ...,p03,0,p17,0,,
233,Verses where the psalmist remembers past deliv...,TFIDF,4,7.08,Psalter,61,Shall not my soul be subject unto God? for fro...,p06,10,,,,
234,Verses where the psalmist remembers past deliv...,TFIDF,5,5.61,Psalter,35,"The transgressor, that he may sin, saith withi...",p10,2,p05,7,,


In [13]:
print(external.columns.tolist())


['Query', 'Method', 'numbered_result', 'Similarity Score (%)', 'Text', 'Psalm Num', 'Verse', 'User1', 'Score1', 'User2', 'Score2', 'User3', 'Score3']


In [14]:
# Unpivot User/Score pairs
df_long = pd.wide_to_long(
    external,
    stubnames=["User", "Score"],  # the base column names
    i=["Query", "Method", "numbered_result", "Similarity Score (%)", "Text", "Psalm Num", "Verse"],  # columns to keep
    j="Pair",  # new column for the pair number
    sep=""      # number comes directly after the stub name
).reset_index()

# Optional: reorder columns
df_long = df_long[["Query", "Method", "numbered_result",  "Similarity Score (%)", "Text", "Psalm Num", "Verse", "Pair", "User", "Score"]]



In [15]:
external = df_long[["Query", "Method", "numbered_result",  "Similarity Score (%)", "Text", "Psalm Num", "Verse", "User", "Score"]]

external.head()

,Query,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,p06,5
1,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,p03,1
2,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,p08,3
3,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,p03,1
4,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,p04,8


In [16]:
caden.head()

,Query,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,caden,9
1,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,caden,6
2,Create in me a clean heart,TFIDF_GLoVe,3,21.90,Bible,31,By David concerning understanding Blessed are ...,caden,3
3,Create in me a clean heart,TFIDF_GLoVe,4,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...,caden,10
4,Create in me a clean heart,TFIDF_GLoVe,5,18.12,Bible,61,For the End for Jeduthun a psalm by David Shal...,caden,7


## Combining the Prepared Data back together

In [17]:
scores = pd.concat([caden, external], ignore_index=True)
scores

,Query,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,caden,9
1,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,caden,6
2,Create in me a clean heart,TFIDF_GLoVe,3,21.90,Bible,31,By David concerning understanding Blessed are ...,caden,3
3,Create in me a clean heart,TFIDF_GLoVe,4,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...,caden,10
4,Create in me a clean heart,TFIDF_GLoVe,5,18.12,Bible,61,For the End for Jeduthun a psalm by David Shal...,caden,7
...,...,...,...,...,...,...,...,...,...
939,Verses where the psalmist remembers past deliv...,TFIDF,5,5.61,Psalter,35,"The transgressor, that he may sin, saith withi...",p05,7
940,Verses where the psalmist remembers past deliv...,TFIDF,5,5.61,Psalter,35,"The transgressor, that he may sin, saith withi...",,
941,Verses where the psalmist remembers past deliv...,TFIDF,6,5.46,Psalter,131,"Lord, remember David and all his meekness; how...",p01,7
942,Verses where the psalmist remembers past deliv...,TFIDF,6,5.46,Psalter,131,"Lord, remember David and all his meekness; how...",p06,0


In [18]:
(scores["Score"].notna() & (scores["Score"] != "")).sum()

765

In [19]:
scores["Query"].unique()

array(['Create in me a clean heart', 'For the Peace of the world',
       'Have mercy on me, O God, have mercy on me. For my soul trusts in Thee, and in the shadow of Thy wings will I hope, until iniquity pass away.',
       'How does the psalmist express trust in God while surrounded by fear and uncertainty?',
       'mercy', 'praise in times of suffering', 'prayer',
       'protection from enemies',
       'Rejoice, O ye heavens, sound the trumpets, ye foundation of the earth, thunder forth gladness, O ye mountains: for behold, Emmanuel to the Cross our sins, and the Giver of Life hath slain death, raising up Adam; for He loveth mankind.',
       'The Lord is my shepherd',
       'Verses where the psalmist remembers past deliverance and uses it to find hope in present trials.'],
      dtype=object)

We now have our intended 944 rows of data we can move on to do last few preperations for the analysis. 

# Further Data Cleaning <a href="further_cleaning"></a>

I now need to work on some of the anaiysis of the data and alot of the imediate processiong was done within another notebook so it will be coppied into here. 

## Query Categoization

In [20]:
query_categories = {
    "mercy": "Simple Keyword Queries",
    "prayer":"Simple Keyword Queries",
    "The Lord is my shepherd": "Phrase/Exact Match Queries",
    "Create in me a clean heart":"Phrase/Exact Match Queries",
    "protection from enemies": "Thematic/Semantic Queries",
    "praise in times of suffering": "Thematic/Semantic Queries",
    "How does the psalmist express trust in God while surrounded by fear and uncertainty?":
        "Long/Complex Queries",
    "Verses where the psalmist remembers past deliverance and uses it to find hope in present trials.":
        "Long/Complex Queries",
    "Rejoice, O ye heavens, sound the trumpets, ye foundation of the earth, thunder forth gladness, O ye mountains: for behold, Emanuel to the Cross our sins, and the Giver of Life hath slain death, raising up Adam; for He loveth mankind.":
        "Orthodox Service Quotes",
    "Have mercy on me, O God, have mercy on me. For my soul trusts in Thee, and in the shadow of Thy wings will I hope, until iniquity pass away.":
        "Orthodox Service Quotes",
    "For the Peace of the world": "Orthodox Service Quotes"

}

In [21]:
scores["Query Category"] = scores["Query"].map(query_categories)
scores

,Query,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User,Score,Query Category
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,caden,9,Phrase/Exact Match Queries
1,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,caden,6,Phrase/Exact Match Queries
2,Create in me a clean heart,TFIDF_GLoVe,3,21.90,Bible,31,By David concerning understanding Blessed are ...,caden,3,Phrase/Exact Match Queries
3,Create in me a clean heart,TFIDF_GLoVe,4,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...,caden,10,Phrase/Exact Match Queries
4,Create in me a clean heart,TFIDF_GLoVe,5,18.12,Bible,61,For the End for Jeduthun a psalm by David Shal...,caden,7,Phrase/Exact Match Queries
...,...,...,...,...,...,...,...,...,...,...
939,Verses where the psalmist remembers past deliv...,TFIDF,5,5.61,Psalter,35,"The transgressor, that he may sin, saith withi...",p05,7,Long/Complex Queries
940,Verses where the psalmist remembers past deliv...,TFIDF,5,5.61,Psalter,35,"The transgressor, that he may sin, saith withi...",,,Long/Complex Queries
941,Verses where the psalmist remembers past deliv...,TFIDF,6,5.46,Psalter,131,"Lord, remember David and all his meekness; how...",p01,7,Long/Complex Queries
942,Verses where the psalmist remembers past deliv...,TFIDF,6,5.46,Psalter,131,"Lord, remember David and all his meekness; how...",p06,0,Long/Complex Queries


In [22]:
# reordering the columns of the dataframe
scores = scores [["Query", "Query Category", "Method", "numbered_result", "Similarity Score (%)", "Text", "Psalm Num", 
                  "Verse", "User", "Score" ]]

scores

,Query,Query Category,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,caden,9
1,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,caden,6
2,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,3,21.90,Bible,31,By David concerning understanding Blessed are ...,caden,3
3,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,4,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...,caden,10
4,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,5,18.12,Bible,61,For the End for Jeduthun a psalm by David Shal...,caden,7
...,...,...,...,...,...,...,...,...,...,...
939,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5,5.61,Psalter,35,"The transgressor, that he may sin, saith withi...",p05,7
940,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5,5.61,Psalter,35,"The transgressor, that he may sin, saith withi...",,
941,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,6,5.46,Psalter,131,"Lord, remember David and all his meekness; how...",p01,7
942,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,6,5.46,Psalter,131,"Lord, remember David and all his meekness; how...",p06,0


In [23]:
# reordering the columns of the dataframe
scores = scores [["Query", "Query Category", "Method","Similarity Score (%)", "numbered_result",
                  "Text", "Psalm Num", "Verse", "User", "Score" ]]

scores

,Query,Query Category,Method,Similarity Score (%),numbered_result,Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,26.10,1,Psalter,100,I will sing of mercy and judgment unto Thee O ...,caden,9
1,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,25.67,2,Bible,4,For the End in psalms an ode by David You hear...,caden,6
2,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,21.90,3,Bible,31,By David concerning understanding Blessed are ...,caden,3
3,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.33,4,Psalter,31,Blessed are they whose iniquities are forgiven...,caden,10
4,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.12,5,Bible,61,For the End for Jeduthun a psalm by David Shal...,caden,7
...,...,...,...,...,...,...,...,...,...,...
939,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he may sin, saith withi...",p05,7
940,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he may sin, saith withi...",,
941,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5.46,6,Psalter,131,"Lord, remember David and all his meekness; how...",p01,7
942,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5.46,6,Psalter,131,"Lord, remember David and all his meekness; how...",p06,0


In [24]:
# filtering to only be studying the top 5 results from each query
scores = scores[scores['numbered_result'] != 6]


pd.set_option("display.max_rows", 50)

scores[scores['Method'] == 'TFIDF']

,Query,Query Category,Method,Similarity Score (%),numbered_result,Text,Psalm Num,Verse,User,Score
16,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF,15.05,1,Psalter,50,"Have mercy upon me, O God, according to Thy gr...",caden,10
17,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF,14.21,2,Psalter,54,"Give ear to my prayer, O God, and despise not ...",caden,9
18,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF,14.20,3,Psalter,23,"The earth is the Lord's, and the fulness there...",caden,6
19,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF,12.53,4,Bible,50,For the End a psalm by David 2when Nathan the ...,caden,8
20,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF,12.48,5,Psalter,40,Blessed is he that considereth the poor and ne...,caden,4
...,...,...,...,...,...,...,...,...,...,...
936,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,7.08,4,Psalter,61,Shall not my soul be subject unto God? for fro...,,
937,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,7.08,4,Psalter,61,Shall not my soul be subject unto God? for fro...,,
938,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he may sin, saith withi...",p10,2
939,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he may sin, saith withi...",p05,7


In [25]:
external = external[external['numbered_result'] != 6]

---

# Analysis

## Inner-Annotator Agreement
Every person that contributed to the scoring of the results aproached them differently even though the same intetion was behind each score conceived. Each result of the *236 results*, was scored up to three times. This may cause discrepency in how each result was scored overall. **Inner-Anotator Agreement** works at trying to normalize the differeint in scored overall, and for each indivual result. 
There are a few different metricsx that handel this. The data for this study is ordinal which means that a `1` is closer to `2` than `5`, making this just just a category. For this reason **Krippendorff's** alpha agreement is what's going to be used. 

The overall metric consists of the following equation:
$$
\alpha = 1 - \frac{D_0}{D_e}
$$

Where $D_0$ is:
$$
D_o = \frac{1}{n} \sum_{c} \sum_{k} o_{ck} \, \delta_{ck}^2
$$

And $D_e$ is:
$$
D_e = \frac{1}{n(n-1)} \sum_{c} \sum_{n_c} n_c * n_{k\ ordinal} \, \delta_{ck}^2
$$

In [26]:
import pandas as pd
import numpy as np

Code for: $$D_o = \frac{1}{n} \sum_{c} \sum_{k} o_{ck} \, \delta_{ck}^2$$

In [27]:
import numpy as np

o_ck = [[0.0]*11 for _ in range(11)]
pairable_n = 0

o_ck

[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]

Building the $o_ck$ portion of $D_0$:

In [28]:
pairable_n = 0

# coincidence matrix (0–10 scale)
o_ck = [[0.0]*11 for _ in range(11)]

# marginal frequencies n_c
counts = [0]*11

In [29]:
def process_row_external(row):

    scores = [row['Score1'], row['Score2'], row['Score3']]

    valid_scores = []
    for s in scores:
        if not pd.isna(s) and s != "":
            val = int(float(s))
            valid_scores.append(val)
            counts[val] += 1   # <-- THIS is how you get counts

    m_u = len(valid_scores)

    if m_u < 2:
        return

    global pairable_n
    pairable_n += m_u

    for i in range(m_u):
        for j in range(m_u):
            if i == j:
                continue
            c = valid_scores[i]
            k = valid_scores[j]
            o_ck[c][k] += 1/(m_u-1)

In [30]:
for _, row in df.iterrows():
    process_row_external(row)

print(counts)

[48, 63, 49, 43, 35, 46, 49, 35, 63, 49, 49]


<div style=" padding:10px; display:inline-block;">
$$
\delta^2_{ck} = \left( \sum_{g=c}^{\max(c,g)} n_g - \frac{n_c + n_k}{2} \right)^2
$$
</div>

In [31]:
def ordinal_delta_sq(c, k):
    if c == k:
        return 0.0
    
    low = min(c, k)
    high = max(c, k)

    # cumulative counts between ranks
    cumu = sum(counts[g] for g in range(low, high+1))

    # subtract half endpoints
    cumu -= (counts[c] + counts[k]) / 2

    return cumu ** 2

In [32]:
def compute_D_o():
    total_coincidences = sum(sum(row) for row in o_ck)
    if total_coincidences == 0:
        return None  # or 0

    total = 0
    for c in range(len(o_ck)):
        for k in range(len(o_ck)):
            delta_sq = ordinal_delta_sq(c, k)
            total += o_ck[c][k] * delta_sq

    return total / total_coincidences

In [33]:
pairable_n = 0
o_ck = [[0.0]*11 for _ in range(11)]
counts = [0]*11  # important if using ordinal_delta_sq

for idx, row in df.iterrows():
    process_row_external(row)

D_o = compute_D_o()

D_o

32326.58969072165

    d_0(temp.iloc[0]
### Building the Coincidences Matrix

In [34]:
o_ck

[[12.0, 6.0, 7.5, 4.5, 1.5, 3.0, 4.5, 3.0, 2.5, 2.5, 0.0],
 [6.0, 9.0, 7.5, 3.0, 3.0, 5.5, 4.0, 4.0, 8.5, 3.0, 3.5],
 [7.5, 7.5, 4.0, 5.0, 3.0, 3.5, 2.0, 4.5, 4.0, 3.0, 2.0],
 [4.5, 3.0, 5.0, 3.0, 4.5, 3.0, 5.0, 2.0, 5.5, 1.5, 1.0],
 [1.5, 3.0, 3.0, 4.5, 1.0, 4.5, 4.5, 1.0, 4.0, 1.0, 1.0],
 [3.0, 5.5, 3.5, 3.0, 4.5, 5.0, 4.0, 2.0, 3.5, 5.0, 3.0],
 [4.5, 4.0, 2.0, 5.0, 4.5, 4.0, 7.0, 3.5, 4.5, 4.0, 3.0],
 [3.0, 4.0, 4.5, 2.0, 1.0, 2.0, 3.5, 4.0, 1.5, 3.0, 4.5],
 [2.5, 8.5, 4.0, 5.5, 4.0, 3.5, 4.5, 1.5, 7.0, 7.5, 8.5],
 [2.5, 3.0, 3.0, 1.5, 1.0, 5.0, 4.0, 3.0, 7.5, 7.0, 7.5],
 [0.0, 3.5, 2.0, 1.0, 1.0, 3.0, 3.0, 4.5, 8.5, 7.5, 11.0]]

### $D_e$
Code for: $D_e = \frac{1}{n(n-1)} \sum_{c} \sum_{n_c} n_c * n_{k\ metric} \, \delta_{ck}^2$

*Where*:
> - $n_c$ = number of times score `c` occurs in the dataset  
> - $n_k$ = number of times score `k` occurs in the dataset  
> - $\delta_{ck}^2$ = squared distance between scores `c` and `k`  
> - `n(n-1)` = total number of pairs in the dataset

Code for 
$$
D_e = \frac{\sum_c \sum_k n_c n_k \left( \sum_{g=\min(c,k)}^{\max(c,k)} n_g - \frac{n_c + n_k}{2} \right)^2}{n(n-1)}, 
\quad n = \sum_c n_c
$$

In [35]:
def compute_D_e():

    n = sum(counts)

    total = 0
    for c in range(11):
        for k in range(11):
            delta_sq = ordinal_delta_sq(c, k)
            total += counts[c] * counts[k] * delta_sq

    return total / (n*(n-1))

Now we can refer to the original formula: $\alpha = 1 - \frac{D_0}{D_e}$

In [36]:
def alpha():
    d_o = compute_D_o()
    d_e = compute_D_e()
    print("D_e = " + str(d_e))
    return 1 - (d_o / d_e)

In [37]:
external.head()

,Query,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,p06,5
1,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,p03,1
2,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,p08,3
3,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,p03,1
4,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,p04,8


In [38]:
alpha()

D_e = 46304.29734848485


0.301866316047674

- `02/20/2026`- $\alpha = 0.614410708025859$
    - I was not properly computing $D_e$ correctly. I was using $(c-k)^2$ rather than $\delta^2_{ck}$. The code above represents the ordinal computations.
- `02/21/2026`- $\alpha = 0.9999895570896626$
Debugging

## Testing with the offical python package


Confirming my results with the offical pacakge before going further. 

In [39]:
# %pip install krippendorff
import krippendorff

In [40]:
import numpy as np
import pandas as pd

temp = df

cols_without = ['Score1', 'Score2', 'Score3']
cols_with = ['CadenScore', 'Score1', 'Score2', 'Score3']

temp[cols_without] = temp[cols_without].apply(pd.to_numeric, errors='coerce')
temp[cols_with] = temp[cols_with].apply(pd.to_numeric, errors='coerce')

In [41]:
data_without_caden = temp[['Score1', 'Score2', 'Score3']].T.to_numpy()

alpha_without = krippendorff.alpha(
    reliability_data=data_without_caden,
    level_of_measurement='ordinal'
)

print("Without Caden:", alpha_without)

Without Caden: 0.3118507320441355


# Rebuilding

Resuilding the metric computation to re compute the score to be able to compute quickly for specfric sets of the data. 

In [42]:
def build_coincidence_matrix(df, score_columns, max_score=10):

    counts = [0]*(max_score+1)
    o_ck = np.zeros((max_score+1, max_score+1))
    pairable_n = 0

    for _, row in df.iterrows():
        scores = [row[col] for col in score_columns]
        
        valid_scores = []
        for s in scores:
            if not pd.isna(s) and s != "":
                val = int(float(s))
                valid_scores.append(val)
                counts[val] += 1

        m_u = len(valid_scores)
        if m_u < 2:
            continue

        pairable_n += m_u

        for i in range(m_u):
            for j in range(m_u):
                if i == j:
                    continue
                c = valid_scores[i]
                k = valid_scores[j]
                o_ck[c][k] += 1/(m_u-1)

    return counts, o_ck, pairable_n

In [43]:
def ordinal_delta_sq(c, k, counts):
    if c == k:
        return 0.0

    low = min(c, k)
    high = max(c, k)

    cumu = sum(counts[g] for g in range(low, high+1))
    cumu -= (counts[c] + counts[k]) / 2

    return cumu ** 2

In [44]:
def compute_D_o(o_ck, counts, max_score=10):

    total_coincidences = o_ck.sum()
    if total_coincidences == 0:
        return None

    total = 0
    for c in range(max_score+1):
        for k in range(max_score+1):
            delta_sq = ordinal_delta_sq(c, k, counts)
            total += o_ck[c][k] * delta_sq

    return total / total_coincidences

In [45]:
def compute_D_e(counts, max_score=10):

    n_total = sum(counts)
    if n_total < 2:
        return None

    total = 0
    for c in range(max_score+1):
        for k in range(max_score+1):
            delta_sq = ordinal_delta_sq(c, k, counts)
            total += counts[c] * counts[k] * delta_sq

    return total / (n_total * (n_total - 1))

In [46]:
def compute_alpha(D_o, D_e):

    if D_o is None or D_e is None:
        return None

    if D_e == 0:
        return 1.0

    return 1 - (D_o / D_e)

In [47]:
def krippendorff_alpha_ordinal(df, score_columns, max_score=10):

    counts, o_ck, pairable_n = build_coincidence_matrix(
        df, score_columns, max_score
    )

    D_o = compute_D_o(o_ck, counts, max_score)
    D_e = compute_D_e(counts, max_score)

    alpha = compute_alpha(D_o, D_e)

    return alpha #, D_o, D_e, counts, o_ck

With these functions built, I want to veriofy they are working the way they did before. 

In [48]:
krippendorff_alpha_ordinal(df, ['Score1', 'Score2', 'Score3'], max_score=10)

0.301866316047674

We are getting the same score a before. This is an easier way of applying the metic because it is easier to change what data to compute. Lets. work on trying different parts of the data. 

### Adding my score

In [49]:
krippendorff_alpha_ordinal(df, ['CadenScore', 'Score1', 'Score2', 'Score3'], max_score=10)

0.27234941428114334

### Looking at each indivual Algorithm
#### TFIDF

In [50]:
temp = df[df['Method'] == 'TFIDF']

print("Witout Caden: ",
      (krippendorff_alpha_ordinal(temp, ['Score1', 'Score2', 'Score3'], max_score=10)))


print("With Caden: ", 
      (krippendorff_alpha_ordinal(temp, ['CadenScore', 'Score1', 'Score2', 'Score3'], max_score=10)))


Witout Caden:  0.34245197162155794
With Caden:  0.21217440217819983


#### TFIDF x GLoVE

In [51]:
temp = df[df['Method'] == 'TFIDF_GLoVe']

print("Witout Caden: ",
      (krippendorff_alpha_ordinal(temp, ['Score1', 'Score2', 'Score3'], max_score=10)))


print("With Caden: ", 
      (krippendorff_alpha_ordinal(temp, ['CadenScore', 'Score1', 'Score2', 'Score3'], max_score=10)))


Witout Caden:  0.29833979878805117
With Caden:  0.36664310187689797


#### BERT

In [52]:
temp = df[df['Method'] == 'BERT']

print("Witout Caden: ",
      (krippendorff_alpha_ordinal(temp, ['Score1', 'Score2', 'Score3'], max_score=10)))


print("With Caden: ", 
      (krippendorff_alpha_ordinal(temp, ['CadenScore', 'Score1', 'Score2', 'Score3'], max_score=10)))


Witout Caden:  0.23981428563451446
With Caden:  0.226705514569417


#### SBERT

In [53]:
temp = df[df['Method'] == 'SBERT']

print("Witout Caden: ",
      (krippendorff_alpha_ordinal(temp, ['Score1', 'Score2', 'Score3'], max_score=10)))


print("With Caden: ", 
      (krippendorff_alpha_ordinal(temp, ['CadenScore', 'Score1', 'Score2', 'Score3'], max_score=10)))


Witout Caden:  0.30125837074104056
With Caden:  0.2194246709308254


## Investigating the Low Alpha Score

The overall `Krippendorff Aplpha` score I cdalculated was
$$\alpha_{w/out\ Caden} = 0.3132363233535458$$
and 
$$\alpha_{w/ \ Caden} = 0.27457889178612493$$

These tell us that the scores between exyone is not reliable, Lets see what might be contributing to that. 

### Looking at the everage score of each user

In [54]:
scores.head()

,Query,Query Category,Method,Similarity Score (%),numbered_result,Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,26.10,1,Psalter,100,I will sing of mercy and judgment unto Thee O ...,caden,9
1,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,25.67,2,Bible,4,For the End in psalms an ode by David You hear...,caden,6
2,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,21.90,3,Bible,31,By David concerning understanding Blessed are ...,caden,3
3,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.33,4,Psalter,31,Blessed are they whose iniquities are forgiven...,caden,10
4,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.12,5,Bible,61,For the End for Jeduthun a psalm by David Shal...,caden,7


In [55]:
import pandas as pd

# Convert 'Score' to numeric, invalid parsing becomes NaN
scores['Score'] = pd.to_numeric(scores['Score'], errors='coerce')

# Optional: drop rows where conversion failed
scores = scores.dropna(subset=['Score'])

# Finally, convert to integer
scores['Score'] = scores['Score'].astype(int)

/tmp/ipykernel_167/1456537010.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  scores['Score'] = pd.to_numeric(scores['Score'], errors='coerce')


In [56]:
scores.pivot_table(index='User', values='Score', aggfunc='mean')

,Score
User,
caden,5.986047
p01,5.042553
p02,6.041667
p03,3.388235
p04,5.809524
p05,8.096774
p06,4.377778
p07,6.272727
p08,6.322034


## Weighted Fleiss' Kappa: An extension of Cohen's Kappa for three or more annotators.

In [57]:
%pip install pyglet

Note: you may need to restart the kernel to use updated packages.


In [59]:
import numpy as np
from statsmodels.stats.inter_rater import fleiss_kappa
from pygwet import gwet_ac1

# Helper: convert score dataframe columns to categorical ratings
def scores_to_categories(df, columns, max_score=10):
    """
    df: pandas DataFrame
    columns: list of columns to include
    max_score: maximum possible score
    """
    # Round scores to nearest integer for categorical agreement
    ratings = df[columns].round().astype(int)
    return ratings

# Step 1: Prepare ratings
columns_without_caden = ['Score1', 'Score2', 'Score3']
columns_with_caden = ['CadenScore', 'Score1', 'Score2', 'Score3']

ratings_wo_caden = scores_to_categories(temp, columns_without_caden)
ratings_w_caden = scores_to_categories(temp, columns_with_caden)

# Step 2: Fleiss's Kappa
def fleiss_from_ratings(ratings, max_score=10):
    """
    Convert ratings (items x raters) into counts per category for Fleiss's Kappa
    """
    data = []
    for _, row in ratings.iterrows():
        counts = [0]*max_score
        for r in row:
            counts[r-1] += 1  # category index = score-1
        data.append(counts)
    return np.array(data)

data_wo_caden = fleiss_from_ratings(ratings_wo_caden)
data_w_caden = fleiss_from_ratings(ratings_w_caden)

kappa_wo_caden = fleiss_kappa(data_wo_caden)
kappa_w_caden = fleiss_kappa(data_w_caden)

print("Fleiss's Kappa without Caden:", kappa_wo_caden)
print("Fleiss's Kappa with Caden:", kappa_w_caden)

# Step 3: Gwet's AC1
def ac1_from_ratings(ratings):
    # Convert DataFrame to list of lists (items x raters)
    rating_list = ratings.values.tolist()
    ac1, se, ci_low, ci_high = gwet_ac1(rating_list)
    return ac1

ac1_wo_caden = ac1_from_ratings(ratings_wo_caden)
ac1_w_caden = ac1_from_ratings(ratings_w_caden)

print("Gwet's AC1 without Caden:", ac1_wo_caden)
print("Gwet's AC1 with Caden:", ac1_w_caden)

ModuleNotFoundError: No module named 'pygwet'

---
---
        def d_e(row):
        # Step 1: Get all scores
        scores = [row['Score1'], row['Score2'], row['Score3']]

        # Step 2: Keep only non-NaN scores
        valid_scores = [float(s) for s in scores if not pd.isna(s)]
        n = len(valid_scores)

        if n < 2:
            return 0  # no disagreement possible

        # Step 3: Compute weighted squared differences using your counts array
        seen = set()
        weighted_squared_diffs = []

        for i in range(n):
            for j in range(i+1, n):  # j > i avoids self-pairs
                pair = (valid_scores[i], valid_scores[j])
                # sort the pair so (1,3) == (3,1)
                pair_sorted = tuple(sorted(pair))

                if pair_sorted in seen:
                    continue
                seen.add(pair_sorted)

                c, k = pair_sorted
                delta_sq = (c - k) ** 2
                contribution = counts[int(c)] * counts[int(k)] * delta_sq
                weighted_squared_diffs.append(contribution)

                # Print like d_0
                print(f"Comparing score {i} ({c}) and score {j} ({k}): squared diff = {delta_sq}, contribution = {contribution}")

        D_e_row = sum(weighted_squared_diffs) / (n * (n - 1))

        return D_e_row


In [ ]:
df = temp.head(1)
df

# Wrong code before fully understanding the metric
    def d_0(row):
        # Step 1: Get all the scores
        scores = [row['Score1'], row['Score2'], row['Score3']]

        # Step 2: Keep only non-NaN scores
        valid_scores = []
        for s in scores:
            if not pd.isna(s):
                valid_scores.append(float(s))

        n = len(valid_scores)

        # Step 3: If fewer than 2 scores, no disagreement
        if n < 2:
            return 0

        # Step 4: Compute all pairwise squared differences
        squared_diffs = []
        for i in range(n):
            for j in range(i+1, n):  # only j>i to avoid duplicates and self-pairs
                diff = valid_scores[i] - valid_scores[j]
                squared_diff = diff ** 2
                squared_diffs.append(squared_diff)
                print(f"Comparing score {i} ({valid_scores[i]}) and score {j} ({valid_scores[j]}): squared diff = {squared_diff}")

        # Step 5: Compute average squared difference
        D_o_row = sum(squared_diffs) / len(squared_diffs)

        return D_o_row
